# Evaluate Gemini on test dataset

This notebook was used to evaluate the Gemini model on the test dataset. You need your own API key to run this.

In [ ]:
from google import genai
import pandas as pd
import time
import traceback
import csv

test_dataset = pd.read_csv("test_dataset.csv")

In [ ]:
start_idx, end_idx = 0, len(test_dataset)

# Add your own API key here
client = genai.Client(api_key="XXX")

file = "eval_results_gemini.csv"
with open(file, "w", encoding="utf-8") as f:
    csv.writer(f).writerow(["prompt","completion","output"])

for i, row in test_dataset.iterrows():
    try:
        prompt = row['prompt']    
        # Don't include the report
        prompt = prompt.split("### Poročilo")[0]    
        instructions = \
"""
Generiraj poročilo o prometu na osnovi vhodnih podatkov, ki se začnejo z '### Vhodni podatki'. Odgovor naj vsebuje samo poročilo. Odgovarjaj v povedih.
Drži se hierarhije dogodkov (od najpomembnejših do najmanj pomembnih): 
- Voznik vozi v napačno smer  
- Zaprta avtocesta 
- Nesreča z zastojem na avtocesti 
- Zastoji zaradi del na avtocesti (ob krajših zastojih se pogosto dogajajo naleti) 
- Zaradi nesreče zaprta glavna ali regionalna cesta 
- Nesreče na avtocestah in drugih cestah 
- Pokvarjena vozila, ko je zaprt vsaj en prometni pas 
- Žival, ki je zašla na vozišče 
- Predmet/razsut tovor na avtocesti 
- Dela na avtocesti, kjer je večja nevarnost naleta (zaprt prometni pas, pred predori, v predorih, …) 
- Zastoj pred Karavankami in mejnimi prehodi 
Pomembno je sporočiti, če voznik ne vozi več v napačno smer ali če je konec zastojev zaradi katere koli prometne nesreče.
"""
        prompt = instructions + "\n\n" + prompt
        response = client.models.generate_content(
            model="gemini-2.0-flash",
            contents=prompt
        )
        
        print("Processed index " + str(i))
        
        with open(file, "a") as f:
            csv.writer(f).writerow([prompt, row["content"], response.text])
        
        # Sleep to avoid hitting API limits
        time.sleep(4)
        
    except Exception:
        print(traceback.format_exc())
